In [1]:
import sys
import torch
import transformers

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Python executable:", sys.executable)

Python: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
PyTorch: 2.13.0+cpu
Transformers: 4.49.0
Python executable: C:\sem5\edi\AllergyGuard\.venv\Scripts\python.exe


# Florence-2 Experiment

## Goal
Load the pretrained Florence-2 model and verify that it works.

## Why?
Florence-2 will provide visual evidence from food images.

## Flow
Image → Florence-2 → Visual evidence

In [2]:
from transformers import AutoProcessor, AutoModelForCausalLM

MODEL_NAME = "microsoft/Florence-2-base"

print("Florence-2 imports successful")

Florence-2 imports successful


In [3]:
print("Loading Florence-2 processor...")

processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

print("Processor loaded successfully!")

Loading Florence-2 processor...
Processor loaded successfully!


## Loading the Florence-2 Model

The processor prepares the input.

The model performs the actual visual understanding.

Flow:

Image + instruction
        ->
    Processor
        ->
Florence-2 Model
        ->
Generated visual evidence

In [4]:
print("Loading Florence-2 model...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

print("Florence-2 model loaded successfully!")

Loading Florence-2 model...
Florence-2 model loaded successfully!


In [5]:
print("Model type:", type(model).__name__)
print("Device:", next(model.parameters()).device)

Model type: Florence2ForConditionalGeneration
Device: cpu


## First Florence-2 Inference

### Goal
Give Florence-2 an image and ask it to describe what it sees.

### Flow
Image
↓
Processor
↓
Florence-2
↓
Generated description

This verifies that the complete vision pipeline works.

In [6]:
from PIL import Image

IMAGE_PATH = "data/raw/test_image.jpg"

image = Image.open(IMAGE_PATH)

print("Image loaded successfully!")
print("Size:", image.size)
print("Mode:", image.mode)

Image loaded successfully!
Size: (464, 280)
Mode: RGB


## First Florence-2 Inference

We give Florence-2 an image and a task prompt.

Task:
`<CAPTION>`

Input:
Image + task

Output:
A generated description of the image.

In [7]:
task_prompt = "<CAPTION>"

inputs = processor(
    text=task_prompt,
    images=image,
    return_tensors="pt"
)

print("Input prepared successfully!")
print(inputs.keys())

Input prepared successfully!
dict_keys(['input_ids', 'attention_mask', 'pixel_values'])


In [8]:
generated_ids = model.generate(
    input_ids=inputs["input_ids"],
    pixel_values=inputs["pixel_values"],
    max_new_tokens=100,
    num_beams=3
)

print("Generation completed!")

Generation completed!


In [9]:
generated_text = processor.batch_decode(
    generated_ids,
    skip_special_tokens=False
)

print(generated_text)

['</s><s>a bowl of chicken and couscous salad on a wooden table</s>']


## Florence-2 Output Processing

The model generates token IDs first.

We then decode and post-process them into a usable result.

Flow:

Model
↓
Generated token IDs
↓
Processor
↓
Structured result

In [10]:
decoded_text = processor.batch_decode(
    generated_ids,
    skip_special_tokens=False
)[0]

print("Raw output:")
print(decoded_text)

Raw output:
</s><s>a bowl of chicken and couscous salad on a wooden table</s>


In [11]:
parsed_result = processor.post_process_generation(
    decoded_text,
    task=task_prompt,
    image_size=(image.width, image.height)
)

print(parsed_result)

{'<CAPTION>': 'a bowl of chicken and couscous salad on a wooden table'}


In [12]:
print("Parsed result:")
print(parsed_result)

print("\nResult type:")
print(type(parsed_result))

Parsed result:
{'<CAPTION>': 'a bowl of chicken and couscous salad on a wooden table'}

Result type:
<class 'dict'>


## Florence-2: Dense Region Captioning

### Goal
Extract multiple visual observations from the image.

Instead of producing one caption, Florence-2 identifies
different regions/objects and describes them.

### Why?
AllergyGuard needs multiple pieces of visual evidence
that can later be compared with OCR and knowledge sources.

Flow:

Image
↓
Florence-2
↓
Multiple visual regions
↓
Visual evidence

In [13]:
task_prompt = "<DENSE_REGION_CAPTION>"

print("Task:", task_prompt)

Task: <DENSE_REGION_CAPTION>


In [14]:
inputs = processor(
    text=task_prompt,
    images=image,
    return_tensors="pt"
)

print("Image prepared for dense region captioning.")

Image prepared for dense region captioning.


In [15]:
generated_ids = model.generate(
    input_ids=inputs["input_ids"],
    pixel_values=inputs["pixel_values"],
    max_new_tokens=200,
    num_beams=3
)

print("Generation completed.")

Generation completed.


In [16]:
decoded_text = processor.batch_decode(
    generated_ids,
    skip_special_tokens=False
)[0]

print(decoded_text)

</s><s>grilled chicken with couscous and vegetables<loc_0><loc_0><loc_998><loc_928>grilled pork tenderloin with quinoa and vegetables<loc_93><loc_42><loc_895><loc_928>sliced lime on plate with vegetables<loc_622><loc_97><loc_794><loc_376>tomato<loc_400><loc_634><loc_514><loc_793><loc_178><loc_592><loc_262><loc_718>cucumber<loc_133><loc_499><loc_203><loc_630><loc_504><loc_753><loc_589><loc_838><loc_327><loc_723><loc_399><loc_822>tomato<loc_606><loc_725><loc_662><loc_815><loc_266><loc_726><loc_344><loc_783><loc_787><loc_412><loc_825><loc_490><loc_152><loc_397><loc_185><loc_474>carrot<loc_497><loc_134><loc_529><loc_188></s>


In [17]:
parsed_result = processor.post_process_generation(
    decoded_text,
    task=task_prompt,
    image_size=(image.width, image.height)
)

print(parsed_result)

{'<DENSE_REGION_CAPTION>': {'bboxes': [[0.23199999332427979, 0.14000000059604645, 463.3039855957031, 259.9800109863281], [43.38399887084961, 11.899999618530273, 415.5119934082031, 259.9800109863281], [288.8399963378906, 27.299999237060547, 368.6479797363281, 105.41999816894531], [185.83200073242188, 177.66000366210938, 238.72799682617188, 222.1800079345703], [82.8239974975586, 165.89999389648438, 121.79999542236328, 201.1800079345703], [61.94399642944336, 139.86000061035156, 94.42399597167969, 176.5399932861328], [234.08799743652344, 210.97999572753906, 273.5279846191406, 234.77999877929688], [151.95999145507812, 202.5800018310547, 185.3679962158203, 230.3000030517578], [281.4159851074219, 203.13999938964844, 307.3999938964844, 228.33999633789062], [123.65599822998047, 203.4199981689453, 159.84799194335938, 219.3800048828125], [365.3999938964844, 115.5, 383.031982421875, 137.33999633789062], [70.75999450683594, 111.30000305175781, 86.0719985961914, 132.86000061035156], [230.83999633789

## Florence-2 Evidence Record

The model output is evidence, not ground truth.

We store:
- model name
- task
- observations
- bounding boxes

This structured representation will later be consumed by
the Adaptive Evidence Fusion Engine.

In [18]:
florence_evidence = {
    "source": "florence-2",
    "model": MODEL_NAME,
    "task": task_prompt,
    "observations": parsed_result[task_prompt]["labels"],
    "bboxes": parsed_result[task_prompt]["bboxes"]
}

print(florence_evidence)

{'source': 'florence-2', 'model': 'microsoft/Florence-2-base', 'task': '<DENSE_REGION_CAPTION>', 'observations': ['grilled chicken with couscous and vegetables', 'grilled pork tenderloin with quinoa and vegetables', 'sliced lime on plate with vegetables', 'tomato', 'tomato', 'cucumber', 'cucumber', 'cucumber', 'tomato', 'tomato', 'tomato', 'tomato', 'carrot'], 'bboxes': [[0.23199999332427979, 0.14000000059604645, 463.3039855957031, 259.9800109863281], [43.38399887084961, 11.899999618530273, 415.5119934082031, 259.9800109863281], [288.8399963378906, 27.299999237060547, 368.6479797363281, 105.41999816894531], [185.83200073242188, 177.66000366210938, 238.72799682617188, 222.1800079345703], [82.8239974975586, 165.89999389648438, 121.79999542236328, 201.1800079345703], [61.94399642944336, 139.86000061035156, 94.42399597167969, 176.5399932861328], [234.08799743652344, 210.97999572753906, 273.5279846191406, 234.77999877929688], [151.95999145507812, 202.5800018310547, 185.3679962158203, 230.30

In [19]:
print("Source:", florence_evidence["source"])
print("Task:", florence_evidence["task"])
print("Observations:")

for observation in florence_evidence["observations"]:
    print("-", observation)

Source: florence-2
Task: <DENSE_REGION_CAPTION>
Observations:
- grilled chicken with couscous and vegetables
- grilled pork tenderloin with quinoa and vegetables
- sliced lime on plate with vegetables
- tomato
- tomato
- cucumber
- cucumber
- cucumber
- tomato
- tomato
- tomato
- tomato
- carrot


## Florence-2 OCR Experiment

Goal:
Test whether Florence-2 can extract visible text from an image.

Florence-2 is not our primary OCR system.
PaddleOCR will handle dedicated text extraction.

We are testing this only as an additional evidence source.

In [21]:
task_prompt = "<OCR>"

print("Task:", task_prompt)

Task: <OCR>


In [22]:
inputs = processor(
    text=task_prompt,
    images=image,
    return_tensors="pt"
)

print("Image prepared for OCR.")

Image prepared for OCR.


In [23]:
generated_ids = model.generate(
    input_ids=inputs["input_ids"],
    pixel_values=inputs["pixel_values"],
    max_new_tokens=100,
    num_beams=3
)

print("OCR generation completed.")

OCR generation completed.


In [24]:
decoded_text = processor.batch_decode(
    generated_ids,
    skip_special_tokens=False
)[0]

print("Raw OCR output:")
print(decoded_text)

Raw OCR output:
</s><s>shutterstock.com . 27297775773</s>


In [25]:
parsed_ocr = processor.post_process_generation(
    decoded_text,
    task=task_prompt,
    image_size=(image.width, image.height)
)

print("Parsed OCR result:")
print(parsed_ocr)

Parsed OCR result:
{'<OCR>': 'shutterstock.com . 27297775773'}


## Florence-2 Experiment Findings

### Tested capabilities

1. CAPTION
   - Produces an overall image description.

2. DENSE_REGION_CAPTION
   - Produces multiple visual observations.
   - Provides bounding boxes.
   - Can produce conflicting high-level interpretations.

3. OCR
   - Can extract visible text.
   - The test image contained a Shutterstock watermark rather than useful food-label text.

### Important observation

Florence-2 output should be treated as model-generated evidence,
not ground truth.

A single visual model can produce ambiguous or conflicting
interpretations.

### AllergyGuard role

Florence-2 will primarily provide visual evidence.

PaddleOCR will remain the dedicated OCR component.

Qwen2-VL will provide an independent visual interpretation.

These sources will later be combined by the Adaptive Evidence
Fusion Engine.


In [26]:
print("Florence-2 experiment complete.")
print("Model:", MODEL_NAME)
print("Device:", next(model.parameters()).device)
print("Tested tasks: CAPTION, DENSE_REGION_CAPTION, OCR")

Florence-2 experiment complete.
Model: microsoft/Florence-2-base
Device: cpu
Tested tasks: CAPTION, DENSE_REGION_CAPTION, OCR
